# 07 · PDFs to Chroma with LangGraph: text chunks and image embeddings (standalone)

**Self-contained:** this notebook imports nothing from the capstone repo. Upload this one `.ipynb` to Google Colab and run all cells. It needs no clone and no GitHub token, and no LLM.

**What it builds:** a LangGraph pipeline that reads PDFs and writes **two Chroma collections** into one database folder.

```
load_pdfs -> split_chunks -> store_text -> extract_images -> store_images -> report
```

| Collection | Holds | Embedding model | Vector |
|---|---|---|---|
| `financial_text` | Text chunks from readable pages | `all-MiniLM-L6-v2` through `langchain-huggingface` | 384-d |
| `financial_images` | Embedded images and pages of vector charts | CLIP `clip-ViT-B-32` | 512-d |

**Why two collections:** a Chroma collection holds one vector size, and text and images use different models here. CLIP places images and text in the same space, so a typed question can search the images.

**How images are stored:** the collection holds the CLIP vector, a caption from the page text, metadata and the path of the PNG. The PNG files stay on disk in `IMAGE_DIR`. LangChain's `Chroma.add_images` isn't used because it would put each whole image into the database as base64 text.

**Packages:** `langgraph`, `langchain-core`, `langchain-text-splitters`, `langchain-chroma`, `langchain-huggingface`, `chromadb`, `sentence-transformers`, `pymupdf`. `langchain-community` is left out on purpose: its maintainers are retiring it, so PDF loading is a small `BaseLoader` here.

**Runtime:** a GPU makes embedding faster, but a CPU also works.

**Before running:** set `SEC_USER_AGENT` in cell 1 to your name and e-mail (SEC requires it), or upload the PDFs to `PDF_DIR`. Colab's disk is wiped when the session ends, so set `DB_DIR` and `IMAGE_DIR` to a mounted Drive folder if you want to keep the results.

In [ ]:
# 0. Install packages (Colab only; elsewhere install them yourself)
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf>=1.24", "pillow", "pandas", "langgraph", "langchain-core",
                    "langchain-text-splitters", "langchain-chroma", "langchain-huggingface", "chromadb", "sentence-transformers"], check=True)
print("Colab:", IN_COLAB)

In [ ]:
# 1. Settings
import os
from pathlib import Path

BASE = Path("/content" if IN_COLAB else ".")
PDF_DIR = Path(os.getenv("NB_PDF_DIR", BASE / "financial_pdfs"))
DB_DIR = Path(os.getenv("NB_DB_DIR", BASE / "chroma_financial"))         # the Chroma database folder
IMAGE_DIR = Path(os.getenv("NB_IMAGE_DIR", BASE / "chroma_images"))      # PNG files of the stored images
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "capstone-research your.name@example.com")   # use your real name and e-mail

TEXT_COLLECTION, IMAGE_COLLECTION = "financial_text", "financial_images"
TEXT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
IMAGE_MODEL = "sentence-transformers/clip-ViT-B-32"
CHUNK_SIZE, CHUNK_OVERLAP = 1200, 200

MIN_IMAGE_PIXELS = 100        # skip images narrower or shorter than this (icons, rules)
MIN_IMAGE_PAGE_SHARE = 0.08   # skip images covering less than this share of the page (logos)
VECTOR_PAGE_MIN_SHAPES = 25   # a page with more vector shapes than this is stored whole: usually a chart or a ruled table
MAX_PAGES = int(os.getenv("NB_MAX_PAGES", "0")) or None   # set a small number for a quick trial
BATCH = 128

## 2. The PDFs
Each file is reused if it's already in `PDF_DIR`, otherwise downloaded from SEC EDGAR. The SHA-256 check confirms it's the exact filing. To use your own PDFs, edit this list and remove the `sha256` and `url` values you don't need.

In [ ]:
import hashlib, time, urllib.request

FILINGS = [
    {"company": "AMD", "file": "EDGAR_ARS_AMD_2025.pdf", "sha256": "1a123433f7da9b600a9415ef02211f72cc451928be30bac3dc73c8aca384e664",
     "url": "https://www.sec.gov/Archives/edgar/data/2488/000119312526129106/d46357dars.pdf"},
    {"company": "AbbVie", "file": "EDGAR_ARS_ABBV_2025.pdf", "sha256": "4a921d5ea20c8a3b3d0d1f5ea3868c05633644461f84dceb037148aa374fde24",
     "url": "https://www.sec.gov/Archives/edgar/data/1551152/000110465926033397/tm2532582d2_ars.pdf"},
    {"company": "NVIDIA", "file": "EDGAR_ARS_NVDA_2026.pdf", "sha256": "510a0b90a6ae3751842c5cdcfcf2ca3683e663e718e118c171da190817cfdde1",
     "url": "https://www.sec.gov/Archives/edgar/data/1045810/000104581026000038/a2026-annualxreportxwebxfi.pdf"},
    {"company": "Chevron", "file": "EDGAR_ARS_CVX_2022.pdf", "sha256": "4e90668e7d7314741beb95184d76b3db503ea4d5f17a03f684c5f8850b17af2b",
     "url": "https://www.sec.gov/Archives/edgar/data/93410/000119312523099324/d433226dars.pdf"},
]


def get_pdf(filing):
    path = PDF_DIR / filing["file"]
    if not path.exists():
        if "url" not in filing:
            raise FileNotFoundError(f"Upload {path}")
        if "@" not in SEC_USER_AGENT or "example.com" in SEC_USER_AGENT:
            raise ValueError("Set SEC_USER_AGENT in cell 1 to your name and e-mail, or upload the PDFs to " + str(PDF_DIR))
        PDF_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading", filing["file"])
        request = urllib.request.Request(filing["url"], headers={"User-Agent": SEC_USER_AGENT})
        with urllib.request.urlopen(request, timeout=300) as response:
            path.write_bytes(response.read())
        time.sleep(0.2)                                   # SEC allows at most 10 requests per second
    if filing.get("sha256") and hashlib.sha256(path.read_bytes()).hexdigest() != filing["sha256"]:
        raise ValueError(f"{path.name}: checksum differs from the expected filing")
    return path


PDFS = {f["file"]: {**f, "path": get_pdf(f)} for f in FILINGS}
print("PDF files:", list(PDFS))

## 3. LangChain pieces: loader, splitter, embeddings, Chroma

- `FinancialPDFLoader` is a LangChain `BaseLoader` that yields one `Document` per page, and flags pages whose text layer is garbled (broken font encoding).
- `ClipEmbeddings` is a LangChain `Embeddings` class with `embed_image`, so LangChain's `Chroma` can search images.
- One `chromadb` client serves both collections.

In [ ]:
import re
import chromadb
import pymupdf
import torch
from langchain_chroma import Chroma
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from PIL import Image
from sentence_transformers import SentenceTransformer

STOPWORDS = frozenset("the of and to in a is for on that by with as are be this from or at an our we".split())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def is_garbled(text):
    words = re.findall(r"[a-z]+", text.lower())
    return (len(words) >= 40 and sum(w in STOPWORDS for w in words) / len(words) < 0.05) or text.count("�") > 5


class FinancialPDFLoader(BaseLoader):
    """One Document per PDF page, with file name, company, 1-based page number and a garbled flag."""

    def __init__(self, path, company, max_pages=None):
        self.path, self.company, self.max_pages = Path(path), company, max_pages

    def lazy_load(self):
        with pymupdf.open(self.path) as document:
            for number, page in enumerate(document, 1):
                if self.max_pages and number > self.max_pages:
                    break
                text = page.get_text(sort=True)
                yield Document(page_content=text, metadata={"file": self.path.name, "company": self.company, "page": number,
                                                            "garbled": is_garbled(text)})


class ClipEmbeddings(Embeddings):
    """CLIP through sentence-transformers: images and text share one vector space."""

    def __init__(self, model_name=IMAGE_MODEL):
        self.model = SentenceTransformer(model_name, device=DEVICE)

    def embed_documents(self, texts):
        return self.model.encode(list(texts), batch_size=32, normalize_embeddings=True).tolist()

    def embed_query(self, text):
        return self.embed_documents([text[:300]])[0]        # CLIP's text encoder takes short inputs only

    def embed_image(self, uris):
        vectors = []
        for start in range(0, len(uris), 32):
            images = [Image.open(u).convert("RGB") for u in uris[start:start + 32]]
            vectors += self.model.encode(images, batch_size=32, normalize_embeddings=True).tolist()
        return vectors


text_embedder = HuggingFaceEmbeddings(model_name=TEXT_MODEL, model_kwargs={"device": DEVICE}, encode_kwargs={"normalize_embeddings": True})
image_embedder = ClipEmbeddings()
DB_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(DB_DIR))
text_store = Chroma(client=client, collection_name=TEXT_COLLECTION, embedding_function=text_embedder,
                    collection_metadata={"hnsw:space": "cosine"})
image_collection = client.get_or_create_collection(IMAGE_COLLECTION, metadata={"hnsw:space": "cosine"}, embedding_function=None)
image_store = Chroma(client=client, collection_name=IMAGE_COLLECTION, embedding_function=image_embedder)
print("device:", DEVICE, "| database:", DB_DIR, "| collections:", [c.name for c in client.list_collections()])

## 4. The LangGraph ingestion pipeline

Each node does one job and adds to the shared state. IDs are stable (file, page, position), and items already in the collection are skipped, so running the graph again embeds nothing new.

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph


class IngestState(TypedDict, total=False):
    pages: list
    chunks: list
    images: list
    report: dict


def new_only(collection, ids):
    """The ids not stored yet."""
    existing = set()
    for start in range(0, len(ids), 500):
        existing |= set(collection.get(ids=ids[start:start + 500], include=[])["ids"])
    return [i for i in ids if i not in existing]


def load_pdfs(state):
    pages, report = [], {}
    for name, filing in PDFS.items():
        print("Reading", name)
        loaded = FinancialPDFLoader(filing["path"], filing["company"], MAX_PAGES).load()
        garbled = sum(d.metadata["garbled"] for d in loaded)
        report[name] = {"pages": len(loaded), "garbled_pages": garbled}
        pages += loaded
    return {"pages": pages, "report": report}


def split_chunks(state):
    readable = [d for d in state["pages"] if not d.metadata["garbled"] and d.page_content.strip()]
    for d in readable:
        d.page_content = re.sub(r"\n{3,}", "\n\n", re.sub(r"[ \t]+", " ", d.page_content)).strip()
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks, position = splitter.split_documents(readable), {}
    for chunk in chunks:
        key = (chunk.metadata["file"], chunk.metadata["page"])
        position[key] = position.get(key, -1) + 1
        chunk.id = f"{key[0]}:p{key[1]}:c{position[key]}"
    return {"chunks": chunks}


def store_text(state):
    wanted = {c.id: c for c in state["chunks"]}
    ids = new_only(text_store._collection, list(wanted))
    for start in range(0, len(ids), BATCH):
        batch = ids[start:start + BATCH]
        text_store.add_documents([wanted[i] for i in batch], ids=batch)
        print(f"  text chunks stored: {start + len(batch)}/{len(ids)}", end="\r")
    print(f"\nfinancial_text: {len(ids)} new, {len(wanted) - len(ids)} already stored")
    report = state["report"]
    for name in PDFS:
        report[name]["text_chunks"] = sum(c.metadata["file"] == name for c in state["chunks"])
    report["new_text_chunks"] = len(ids)
    return {"report": report}


def merge_overlapping(boxes):
    """Join rectangles that overlap: designers often slice one picture into slightly overlapping tiles. Charts side by side only share an edge, so they stay separate."""
    boxes = list(boxes)
    merged = True
    while merged:
        merged = False
        for i, a in enumerate(boxes):
            for j in range(i + 1, len(boxes)):
                if a.intersects(boxes[j]):
                    boxes[i] = a | boxes.pop(j)
                    merged = True
                    break
            if merged:
                break
    return boxes


def extract_images(state):
    """Save large embedded images, and whole pages drawn with vector graphics, as PNG files. Repeats within a PDF are kept once."""
    found = []
    for name, filing in PDFS.items():
        seen = set()
        captions = {d.metadata["page"]: "" if d.metadata["garbled"] else re.sub(r"\s+", " ", d.page_content)[:400]
                    for d in state["pages"] if d.metadata["file"] == name}
        with pymupdf.open(filing["path"]) as document:
            for number, page in enumerate(document, 1):
                if MAX_PAGES and number > MAX_PAGES:
                    break
                area = page.rect.get_area() or 1
                boxes = [pymupdf.Rect(i["bbox"]) & page.rect for i in page.get_image_info()
                         if i["width"] >= MIN_IMAGE_PIXELS and i["height"] >= MIN_IMAGE_PIXELS]
                clips = [("embedded_image", b) for b in merge_overlapping(b for b in boxes if not b.is_empty)
                         if b.get_area() / area >= MIN_IMAGE_PAGE_SHARE]
                if not clips and sum(d["rect"].width * d["rect"].height > 4 for d in page.get_drawings()) > VECTOR_PAGE_MIN_SHAPES:
                    clips.append(("vector_graphics_page", page.rect))
                for index, (kind, box) in enumerate(clips):
                    pixmap = page.get_pixmap(matrix=pymupdf.Matrix(1.5, 1.5), clip=box, alpha=False)
                    data = pixmap.tobytes("png")
                    digest = hashlib.sha256(data).hexdigest()
                    if digest in seen:
                        continue
                    seen.add(digest)
                    path = IMAGE_DIR / f"{Path(name).stem}_p{number:04d}_{index}.png"
                    path.write_bytes(data)
                    found.append({"id": f"{name}:p{number}:img{index}", "uri": str(path), "caption": captions.get(number, ""),
                                  "metadata": {"file": name, "company": filing["company"], "page": number, "kind": kind,
                                               "image_path": str(path), "width": pixmap.width, "height": pixmap.height}})
        print(f"{name}: {len(seen)} images")
        state["report"][name]["images"] = len(seen)
    return {"images": found, "report": state["report"]}


def store_images(state):
    wanted = {row["id"]: row for row in state["images"]}
    ids = new_only(image_collection, list(wanted))
    for start in range(0, len(ids), BATCH):
        batch = [wanted[i] for i in ids[start:start + BATCH]]
        image_collection.upsert(ids=[r["id"] for r in batch], uris=[r["uri"] for r in batch],
                                embeddings=image_embedder.embed_image([r["uri"] for r in batch]),
                                documents=[r["caption"] or f"Image from {r['metadata']['file']} page {r['metadata']['page']}" for r in batch],
                                metadatas=[r["metadata"] for r in batch])
        print(f"  images stored: {start + len(batch)}/{len(ids)}", end="\r")
    print(f"\nfinancial_images: {len(ids)} new, {len(wanted) - len(ids)} already stored")
    state["report"]["new_images"] = len(ids)
    return {"report": state["report"]}


def report(state):
    return {"report": {**state["report"], "text_collection_total": text_store._collection.count(),
                       "image_collection_total": image_collection.count()}}


builder = StateGraph(IngestState)
steps = [load_pdfs, split_chunks, store_text, extract_images, store_images, report]
for step in steps:
    builder.add_node(step.__name__ + "_step", step)
builder.add_edge(START, steps[0].__name__ + "_step")
for first, second in zip(steps, steps[1:]):
    builder.add_edge(first.__name__ + "_step", second.__name__ + "_step")
builder.add_edge(steps[-1].__name__ + "_step", END)
ingest = builder.compile()
print(ingest.get_graph().draw_mermaid())

## 5. Run the pipeline
On a CPU this takes several minutes the first time, mostly text embedding and image rendering.

In [ ]:
import pandas as pd


def show(obj):
    try:
        display(obj)
    except NameError:                                     # plain Python, not a notebook
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


started = time.time()
result = ingest.invoke({})
print(f"pipeline finished in {time.time() - started:.0f}s")
show(pd.DataFrame([{"pdf_file": name, **result["report"][name]} for name in PDFS]))
print({k: v for k, v in result["report"].items() if k not in PDFS})

## 6. Check that it persisted, and that a second run adds nothing
A new client reopens the database folder. The graph then runs again and should embed 0 new items.

In [ ]:
reopened = chromadb.PersistentClient(path=str(DB_DIR))
print({c.name: reopened.get_collection(c.name).count() for c in reopened.list_collections()})
sample = reopened.get_collection(IMAGE_COLLECTION).get(limit=1, include=["metadatas", "documents", "uris", "embeddings"])
print("image record:", sample["metadatas"][0], "\n  uri:", sample["uris"][0], "\n  vector size:", len(sample["embeddings"][0]),
      "\n  caption:", sample["documents"][0][:100])
second = ingest.invoke({})["report"]
print("second run: new text chunks", second["new_text_chunks"], "| new images", second["new_images"])

## 7. Search the text collection
Filters keep the search inside one PDF. Each hit shows its file and page.

In [ ]:
TEXT_TESTS = [("What was AMD's diluted earnings per share for fiscal 2025?", "EDGAR_ARS_AMD_2025.pdf", "2.65"),
              ("Who was AbbVie's independent auditor for fiscal year 2025?", "EDGAR_ARS_ABBV_2025.pdf", "Ernst & Young"),
              ("NVIDIA total shareholders' equity fiscal 2026", "EDGAR_ARS_NVDA_2026.pdf", "157,293"),
              ("What was AbbVie's revenue for fiscal 2025?", "EDGAR_ARS_ABBV_2025.pdf", "61,160")]
rows = []
for query, name, gold in TEXT_TESTS:
    hits = text_store.similarity_search_with_score(query, k=5, filter={"file": name})
    rows.append({"query": query[:55], "pdf_file": name, "pages": [d.metadata["page"] for d, _ in hits],
                 "gold_in_top5": any(gold in d.page_content for d, _ in hits), "best_distance": round(hits[0][1], 3) if hits else None})
show(pd.DataFrame(rows))

## 8. Search the image collection
Three kinds of search: **typed text** through CLIP, **typed text within one PDF**, and **search by image**. The target pages are known charts: NVIDIA FY2026 pages 60 and 123, Chevron FY2022 page 35, AMD FY2025 page 59 and AbbVie FY2025 page 164.

In [ ]:
IMAGE_TESTS = [("bar chart comparing revenue with peer companies", "EDGAR_ARS_NVDA_2026.pdf", 60),
               ("bar charts of oil and natural gas production", "EDGAR_ARS_CVX_2022.pdf", 35),
               ("line chart of cumulative total stockholder return", "EDGAR_ARS_AMD_2025.pdf", 59),
               ("line chart of cumulative total stockholder return", "EDGAR_ARS_ABBV_2025.pdf", 164),
               ("line chart of cumulative total stockholder return", "EDGAR_ARS_NVDA_2026.pdf", 123),
               ("chart", "EDGAR_ARS_CVX_2022.pdf", 35)]
rows = []
for query, name, target in IMAGE_TESTS:
    everywhere = [(d.metadata["file"], d.metadata["page"]) for d in image_store.similarity_search(query, k=10)]
    inside = [d.metadata["page"] for d in image_store.similarity_search(query, k=10, filter={"file": name})]
    rows.append({"query": query, "target": f"{name} p{target}",
                 "rank_all_pdfs": everywhere.index((name, target)) + 1 if (name, target) in everywhere else None,
                 "rank_within_pdf": inside.index(target) + 1 if target in inside else None, "top3_within_pdf": inside[:3]})
show(pd.DataFrame(rows))
kinds = pd.Series([m["kind"] for m in image_collection.get(include=["metadatas"])["metadatas"]]).value_counts()
print(kinds.to_string())

stored = image_collection.get(where={"$and": [{"file": "EDGAR_ARS_NVDA_2026.pdf"}, {"page": 60}]}, include=["uris"])
if stored["uris"]:
    uri = stored["uris"][0]
    similar = image_store.similarity_search_by_image(uri, k=4)
    print("search by image", Path(uri).name, "->", [(d.metadata["file"], d.metadata["page"]) for d in similar])
    show(Image.open(uri).resize((480, int(480 * Image.open(uri).height / Image.open(uri).width))))

## What this design can and can't do

| Topic | Behaviour |
|---|---|
| **Finding a figure** | Text search is by meaning only here; Chroma has no built-in keyword search. Exact figures can miss, so check `gold_in_top5` above |
| **Garbled pages** | Their text isn't stored, because this notebook has no OCR, so Chevron's statements can't be found as text. Their images and vector-chart pages can still be stored |
| **Reading numbers in images** | CLIP finds an image that looks related; it doesn't read digits. The value still needs OCR or a vision model |
| **CLIP and documents** | CLIP was trained on photos with captions. A question describing what a chart looks like can work, while a vague one like "chart" often doesn't. Compare the ranks above |
| **CLIP's crop** | CLIP shrinks each image and keeps only a centred 224 x 224 square. It doesn't see the sides of a wide panel or the top and bottom of a whole page, so small labels and footnotes don't count |
| **What gets stored** | Most embedded images in annual reports are photos, covers and decoration. They're stored too, because nothing here tells a photo from a chart. `vector_graphics_page` means a chart **or** a ruled table, stored as a whole page |
| **Moving the database** | `image_path` and the `uri` are absolute paths. If you move the PNG folder, those records point to the old place |
| **One vector per image** | Chroma stores a single vector per record, so it can't run the ColPali multivector MaxSim search of the Qdrant design |
| **Pages with neither** | A table with few or no ruled lines, and no image, gets no image record; its text is in `financial_text`. A page with an embedded image isn't checked for vector charts as well |
| **`text_store._collection`** | LangChain doesn't expose the existence check or the count, so the notebook uses this private attribute. A `langchain-chroma` upgrade could rename it |